# Pipeline: аналитика и подготовка данных до обучения

Notebook собирает этапы из второй главы `database.pdf` и вспомогательных notebooks:

- загрузка телеметрии из `data/{Месяц}/merged_with_mode.csv`;
- базовая аналитика по режимам, пропускам вибрации и шагу времени;
- memory-safe ресемплинг по месяцам без чтения всего года в память;
- обработка коротких пропусков;
- двухэтапная детекция аномалий: Hampel + `IsolationForest` на окнах признаков;
- сохранение минимального prepared CSV для `pipeline_part_2_lib.py`.

Notebook исходит из фиксированного формата проекта:
все месяцы существуют, все входные столбцы есть, разделитель в `merged_with_mode.csv` — запятая.
Запуск выполняет полный годовой прогон и сохраняет итоговый CSV.


## Методические допущения

Из второй главы в pipeline перенесены основные решения:

- в prepared CSV остаются только `timestamp`, `mode`, `power_mode`, `is_target_mode`, `14` каналов вибрации и флаги аномалий;
- временное разбиение только по времени, без случайного shuffle;
- поддержка нескольких временных масштабов через параметр `cfg.resample_rule`;
- короткие пропуски заполняются, длинные не восстанавливаются искусственно;
- детекция аномалий делается в два этапа: локальный робастный фильтр и многомерный `IsolationForest`;
- признаки для моделей, окна, split и lag/rolling table строятся уже во второй части пайплайна.

Если RAM ограничена, начни с `cfg.resample_rule = "1min"`.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
import os
from dotenv import load_dotenv

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)


In [2]:
DATA_DIR = Path("data")

MONTHS_RU = [
    "Январь", "Февраль", "Март", "Апрель", "Май", "Июнь",
    "Июль", "Август", "Сентябрь", "Октябрь", "Ноябрь", "Декабрь",
]

DATE_RAW = "DATE"
TIME_RAW = "TIME"
MODE_RAW = "MODE"
PWR_MODE_RAW = "PWR_MODE"

load_dotenv()
vib_cols_env = os.getenv("VIB_COLS", "").replace("\n", "").split(",")
TARGET_SPECS = []
for i, raw_name in enumerate(vib_cols_env):
    clean_name = raw_name.strip()
    if clean_name:
        alias = f"vib_{i+1:02d}"
        TARGET_SPECS.append((clean_name, alias))

RAW_TO_ALIAS = {
    DATE_RAW: "date",
    TIME_RAW: "time",
    MODE_RAW: "mode",
    PWR_MODE_RAW: "power_mode",
    **dict(TARGET_SPECS),
}

TARGET_COLS = [alias for _, alias in TARGET_SPECS]
ANOMALY_COLS = ["stage1_anomaly", "stage2_anomaly", "stage_any_anomaly"]
PREPARED_COLS = ["timestamp", "mode", "power_mode", "is_target_mode", *TARGET_COLS, *ANOMALY_COLS]
READ_RAW_COLS = [
    DATE_RAW, TIME_RAW, MODE_RAW, PWR_MODE_RAW,
    *[raw for raw, _ in TARGET_SPECS],
]

@dataclass(frozen=True)
class PipelineConfig:
    resample_rule: str = "1s"
    read_chunksize: int = 250_000
    short_gap_limit_steps: int = 6
    hampel_window_seconds: int = 60
    hampel_n_sigmas: float = 3.0
    stage1_min_flag_count: int = 4
    isolation_window_lags: tuple[int, ...] = (0, 1, 2, 3)
    mode_filter: str | None = "В сети"
    isolation_contamination: float = 0.005
    isolation_max_fit_rows: int = 30_000
    random_state: int = 42


cfg = PipelineConfig()

In [3]:
def month_file(month: str) -> Path:
    return DATA_DIR / month / "merged_with_mode.csv"


def build_timestamp(df: pd.DataFrame, date_col: str = "date", time_col: str = "time") -> pd.Series:
    date_part = (
        df[date_col]
        .astype(str)
        .str.replace("/", ".", regex=False)
        .str.strip()
    )
    time_part = (
        df[time_col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )
    combined = date_part + " " + time_part
    parsed = pd.to_datetime(
        combined,
        format="%d.%m.%Y %H:%M:%S.%f",
        errors="coerce",
    )
    missing = parsed.isna()
    if missing.any():
        parsed.loc[missing] = pd.to_datetime(
            combined.loc[missing],
            format="%d.%m.%Y %H:%M:%S",
            errors="coerce",
        )
    return parsed


def standardize_chunk(df: pd.DataFrame) -> pd.DataFrame:
    work = df.rename(columns=RAW_TO_ALIAS).copy()

    for col in TARGET_COLS:
        work[col] = pd.to_numeric(
            work[col].astype(str).str.replace(",", ".", regex=False).str.strip(),
            errors="coerce",
        )

    work["mode"] = work["mode"].astype("string").str.strip()
    work["power_mode"] = work["power_mode"].astype("string").str.strip()
    work["timestamp"] = build_timestamp(work)

    keep_cols = [
        "timestamp", "mode", "power_mode", *TARGET_COLS,
    ]
    work = (
        work[keep_cols]
        .dropna(subset=["timestamp"])
        .sort_values("timestamp")
        .drop_duplicates(subset=["timestamp"], keep="last")
        .reset_index(drop=True)
    )
    return work


def iter_month_chunks(
    month: str,
    usecols: Iterable[str] | None = None,
    chunksize: int = cfg.read_chunksize,
):
    reader = pd.read_csv(
        month_file(month),
        sep=",",
        usecols=usecols or READ_RAW_COLS,
        chunksize=chunksize,
        dtype=str,
        encoding="utf-8",
        low_memory=False,
    )

    for chunk in reader:
        yield chunk


In [4]:
def aggregate_chunk_to_buckets(chunk: pd.DataFrame, rule: str) -> pd.DataFrame:
    work = standardize_chunk(chunk)
    if work.empty:
        return pd.DataFrame()

    work["bucket_ts"] = work["timestamp"].dt.floor(rule)

    grouped = work.groupby("bucket_ts", sort=True)
    meta = grouped.agg(
        bucket_last_ts=("timestamp", "max"),
        mode=("mode", "last"),
        power_mode=("power_mode", "last"),
    )

    num = grouped[TARGET_COLS].agg(["sum", "count"])
    num.columns = [f"{col}__{stat}" for col, stat in num.columns]
    meta = meta.join(num)

    return meta.reset_index()


def combine_bucket_partials(partials: list[pd.DataFrame]) -> pd.DataFrame:
    if not partials:
        return pd.DataFrame(columns=["timestamp", "mode", "power_mode", *TARGET_COLS])

    combined = pd.concat(partials, ignore_index=True).sort_values(["bucket_ts", "bucket_last_ts"])

    agg_map = {
        "bucket_last_ts": "max",
        "mode": "last",
        "power_mode": "last",
    }
    for col in TARGET_COLS:
        sum_col = f"{col}__sum"
        count_col = f"{col}__count"
        agg_map[sum_col] = "sum"
        agg_map[count_col] = "sum"

    out = (
        combined.groupby("bucket_ts", sort=True)
        .agg(agg_map)
        .reset_index()
        .rename(columns={"bucket_ts": "timestamp"})
    )

    for col in TARGET_COLS:
        sum_col = f"{col}__sum"
        count_col = f"{col}__count"
        out[col] = out[sum_col] / out[count_col].replace(0, np.nan)
        out.drop(columns=[sum_col, count_col], inplace=True)

    return out.drop(columns="bucket_last_ts").sort_values("timestamp").reset_index(drop=True)


def aggregate_month_to_rule(
    month: str,
    rule: str = cfg.resample_rule,
    chunksize: int = cfg.read_chunksize,
) -> pd.DataFrame:
    partials = []
    for chunk in iter_month_chunks(month, chunksize=chunksize):
        part = aggregate_chunk_to_buckets(chunk, rule=rule)
        if not part.empty:
            partials.append(part)

    out = combine_bucket_partials(partials)
    return out

In [5]:
def assemble_resampled_dataset(
    months: Iterable[str],
    rule: str = cfg.resample_rule,
    chunksize: int = cfg.read_chunksize,
) -> pd.DataFrame:
    frames = []
    for month in months:
        month_frame = aggregate_month_to_rule(
            month,
            rule=rule,
            chunksize=chunksize,
        )
        if not month_frame.empty:
            frames.append(month_frame)

    if not frames:
        raise RuntimeError("Не удалось собрать ни одного месяца в resampled dataset.")

    out = pd.concat(frames, ignore_index=True)
    out = out.sort_values("timestamp").drop_duplicates("timestamp", keep="last").reset_index(drop=True)
    return out


def reindex_regular_grid(df: pd.DataFrame, rule: str = cfg.resample_rule) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    ordered = df.sort_values("timestamp").set_index("timestamp")
    full_index = pd.date_range(
        ordered.index.min().floor(rule),
        ordered.index.max().ceil(rule),
        freq=rule,
    )
    out = ordered.reindex(full_index).rename_axis("timestamp").reset_index()
    return out


def fill_short_gaps(
    df: pd.DataFrame,
    numeric_cols: Iterable[str],
    category_cols: Iterable[str] = ("mode", "power_mode"),
    max_gap_steps: int = cfg.short_gap_limit_steps,
) -> pd.DataFrame:
    out = df.copy()
    active_numeric = list(numeric_cols)
    missing = out[active_numeric].isna().any(axis=1)
    block_id = missing.ne(missing.shift(fill_value=False)).cumsum()
    gap_size = missing.groupby(block_id).transform("sum")
    short_gap_mask = missing & gap_size.between(1, max_gap_steps)

    interpolated = out[active_numeric].interpolate(method="linear", limit_area="inside")
    out.loc[short_gap_mask, active_numeric] = interpolated.loc[short_gap_mask, active_numeric]

    for col in category_cols:
        out.loc[short_gap_mask, col] = out[col].ffill().loc[short_gap_mask]

    return out

In [6]:
def resample_step_seconds(rule: str = cfg.resample_rule) -> int:
    return int(pd.Timedelta(rule).total_seconds())


def hampel_flags(
    series: pd.Series,
    window_seconds: int = cfg.hampel_window_seconds,
    n_sigmas: float = cfg.hampel_n_sigmas,
    sample_step_seconds: int = resample_step_seconds(),
) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce")
    rolling_window = max(3, int(round(window_seconds / sample_step_seconds)))
    median = x.rolling(window=rolling_window, center=True, min_periods=rolling_window // 2).median()
    abs_dev = (x - median).abs()
    mad = abs_dev.rolling(window=rolling_window, center=True, min_periods=rolling_window // 2).median()
    threshold = 1.4826 * n_sigmas * mad
    return (abs_dev > threshold).fillna(False)


def apply_hampel_stage(
    df: pd.DataFrame,
    cols: Iterable[str] = TARGET_COLS,
    window_seconds: int = cfg.hampel_window_seconds,
    n_sigmas: float = cfg.hampel_n_sigmas,
    min_flag_count: int = cfg.stage1_min_flag_count,
    sample_step_seconds: int = resample_step_seconds(),
) -> pd.DataFrame:
    out = df.copy()
    flags = []
    for col in cols:
        flags.append(hampel_flags(
            out[col],
            window_seconds=window_seconds,
            n_sigmas=n_sigmas,
            sample_step_seconds=sample_step_seconds,
        ))

    out["stage1_anomaly"] = pd.concat(flags, axis=1).sum(axis=1) >= min_flag_count
    return out


def build_isolation_features(
    df: pd.DataFrame,
    feature_cols: Iterable[str] | None = None,
    window_lags: Iterable[int] = cfg.isolation_window_lags,
) -> pd.DataFrame:
    feature_cols = list(feature_cols or TARGET_COLS)
    base = df[feature_cols].copy()

    parts = []
    for lag in window_lags:
        shifted = base.shift(lag).add_suffix(f"_lag_{lag}")
        parts.append(shifted)

    diff_1 = base.diff().add_suffix("_diff_1")
    parts.append(diff_1)

    return pd.concat(parts, axis=1)


def apply_isolation_forest_stage(
    df: pd.DataFrame,
    contamination: float = cfg.isolation_contamination,
    max_fit_rows: int = cfg.isolation_max_fit_rows,
    random_state: int = cfg.random_state,
) -> pd.DataFrame:
    out = df.copy()
    features = build_isolation_features(out)
    valid = features.dropna()

    fit_frame = valid
    if max_fit_rows is not None and len(valid) > max_fit_rows:
        fit_frame = valid.sample(max_fit_rows, random_state=random_state)

    model = IsolationForest(
        contamination=contamination,
        random_state=random_state,
        n_jobs=-1,
    )
    model.fit(fit_frame)

    anomaly = pd.Series(pd.NA, index=out.index, dtype="boolean")
    anomaly.loc[valid.index] = model.predict(valid) == -1

    out["stage2_anomaly"] = anomaly
    out["stage_any_anomaly"] = out["stage1_anomaly"] | anomaly.fillna(False)
    return out

In [7]:
def add_context_features(df: pd.DataFrame, mode_filter: str | None = cfg.mode_filter) -> pd.DataFrame:
    out = df.copy()
    out["is_target_mode"] = True if mode_filter is None else out["mode"].eq(mode_filter)
    return out[PREPARED_COLS]

In [8]:
def run_pretraining_pipeline(
    months: Iterable[str],
    config: PipelineConfig = cfg,
) -> pd.DataFrame:
    base = assemble_resampled_dataset(
        months=months,
        rule=config.resample_rule,
        chunksize=config.read_chunksize,
    )
    base = reindex_regular_grid(base, rule=config.resample_rule)
    prepared = fill_short_gaps(
        base,
        numeric_cols=TARGET_COLS,
        max_gap_steps=config.short_gap_limit_steps,
    )
    prepared = apply_hampel_stage(
        prepared,
        cols=TARGET_COLS,
        window_seconds=config.hampel_window_seconds,
        n_sigmas=config.hampel_n_sigmas,
        min_flag_count=config.stage1_min_flag_count,
        sample_step_seconds=resample_step_seconds(config.resample_rule),
    )
    prepared = apply_isolation_forest_stage(
        prepared,
        contamination=config.isolation_contamination,
        max_fit_rows=config.isolation_max_fit_rows,
        random_state=config.random_state,
    )
    prepared = add_context_features(prepared, mode_filter=config.mode_filter)
    return prepared.sort_values("timestamp").reset_index(drop=True)


def save_prepared_year_dataframe(
    df: pd.DataFrame,
    out_path: Path = Path("data") / "prepared_year_after_preprocessing.csv",
) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df[PREPARED_COLS].to_csv(
        out_path,
        index=False,
        encoding="utf-8",
        date_format="%Y-%m-%d %H:%M:%S.%f",
    )
    return out_path

In [9]:
prepared_year = run_pretraining_pipeline(MONTHS_RU, config=cfg)
_ = save_prepared_year_dataframe(prepared_year)

## Что дальше

Следующий этап уже за пределами этого notebook:

- запустить `pipeline_part_2.ipynb`, который строит окна и обучает SARIMA/VAR/LightGBM/TCN/Transformer.